# SpendWise multilingual categorization benchmark

This notebook benchmarks multilingual semantic category matching. It performs no training or Android export.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
ML_ROOT = cwd if (cwd / 'src' / 'benchmark.py').exists() else cwd / 'ml'
if not (ML_ROOT / 'src' / 'benchmark.py').exists():
    ML_ROOT = cwd.parent
sys.path.insert(0, str(ML_ROOT / 'src'))

from benchmark import MODEL_NAMES, run_benchmark

DATASET_PATH = ML_ROOT / 'data' / 'categorization_benchmark.csv'
OUTPUT_DIR = ML_ROOT / 'data'
data = pd.read_csv(DATASET_PATH)
print(f'{len(data)} examples across {data.category.nunique()} categories')
data.head()

## Dataset distribution

In [ ]:
distribution = data.groupby(['category', 'language']).size().unstack(fill_value=0)
display(distribution)
distribution.plot(kind='bar', stacked=True, figsize=(12, 5))
plt.ylabel('Examples')
plt.title('Benchmark examples by category and language')
plt.tight_layout()
plt.show()

## Run both embedding models

The first execution downloads model files. E5 receives its recommended query and passage prefixes.

In [ ]:
predictions, comparison, group_metrics, errors = run_benchmark(
    DATASET_PATH,
    OUTPUT_DIR,
    model_names=MODEL_NAMES,
    batch_size=32,
)
display(comparison)

## Accuracy by language, OCR variant, and category

In [ ]:
for dimension in ['language', 'variant_type', 'category']:
    print(f'\n{dimension}')
    display(group_metrics[group_metrics.dimension == dimension].pivot(
        index='group', columns='model', values='top1_accuracy'
    ))

## Confusion and error examples

In [ ]:
for model_name in predictions.model.unique():
    print(model_name)
    model_predictions = predictions[predictions.model == model_name]
    confusion = pd.crosstab(
        model_predictions.category,
        model_predictions.predicted_category,
        rownames=['expected'],
        colnames=['predicted'],
    )
    display(confusion)
display(errors.head(25))

## Confidence-gap analysis

Low `top1 - top2` gaps identify ambiguous predictions and can inform a future abstention threshold.

In [ ]:
gap_summary = predictions.groupby(['model', 'top1_correct']).score_gap.agg(['count', 'mean', 'median'])
display(gap_summary)
low_confidence = predictions.sort_values('score_gap').loc[:, [
    'text', 'category', 'predicted_category', 'top1_score', 'top2_category',
    'top2_score', 'score_gap', 'language', 'variant_type', 'model'
]]
display(low_confidence.head(30))
predictions.boxplot(column='score_gap', by=['model', 'top1_correct'], figsize=(12, 5), rot=20)
plt.suptitle('')
plt.title('Confidence gaps for correct and incorrect predictions')
plt.ylabel('Top-1 minus Top-2 cosine similarity')
plt.tight_layout()
plt.show()

## Custom category verification: Books

Books participates only because its bilingual category description was added. No retraining or output-layer change is used.

In [ ]:
books = predictions[predictions.category == 'Books']
books_result = books.groupby('model').agg(
    examples=('text', 'size'),
    top1_accuracy=('top1_correct', 'mean'),
    top3_accuracy=('top3_correct', 'mean'),
).reset_index()
display(books_result)
display(books[['text', 'predicted_category', 'top1_score', 'top2_category', 'score_gap', 'model']])

## Final comparison

In [ ]:
final_comparison = comparison[[
    'model', 'overall_top1_accuracy', 'overall_top3_accuracy',
    'average_inference_ms_per_item', 'model_load_seconds',
    'embedding_dimension', 'cached_model_size_bytes',
    'mean_score_gap_correct', 'mean_score_gap_incorrect',
    'score_gap_correctness_correlation'
]].sort_values(['overall_top1_accuracy', 'overall_top3_accuracy'], ascending=False)
display(final_comparison)

# V2: leakage-safe category prototypes

V2 preserves the description baseline and adds leave-one-canonical-product-out example centroids and 50/50 hybrid prototypes.

In [ ]:
from benchmark_v2 import run_v2_benchmark

V2_DATASET_PATH = ML_ROOT / 'data' / 'categorization_benchmark_v2.csv'
v2_predictions, v2_results, threshold_results, custom_category_results, v2_errors = run_v2_benchmark(
    V2_DATASET_PATH,
    OUTPUT_DIR,
    model_names=MODEL_NAMES,
    batch_size=32,
)

## All six model and prototype configurations

In [ ]:
v2_summary_columns = [
    'model', 'strategy', 'overall_top1_accuracy', 'overall_top3_accuracy',
    'english_top1_accuracy', 'arabic_top1_accuracy', 'mixed_top1_accuracy',
    'clean_top1_accuracy', 'ocr_noise_top1_accuracy', 'average_inference_ms_per_item',
]
display(v2_results[v2_summary_columns].sort_values('overall_top1_accuracy', ascending=False))

## Focus category comparison

In [ ]:
focus_columns = {
    'category_top1__medicine': 'Medicine',
    'category_top1__household': 'Household',
    'category_top1__groceries': 'Groceries',
    'category_top1__restaurants': 'Restaurants',
    'category_top1__books': 'Books',
}
focus_comparison = v2_results[['model', 'strategy', *focus_columns]].rename(columns=focus_columns)
display(focus_comparison)

## Confidence and coverage

The table searches for high-precision operating points without selecting a production threshold.

In [ ]:
safe_thresholds = threshold_results[
    (threshold_results.accepted_accuracy >= 0.90) &
    (threshold_results.accepted_suggestions > 0)
].sort_values(['coverage_percentage', 'accepted_accuracy'], ascending=False)
display(safe_thresholds.head(30))

gap_only = threshold_results[threshold_results.threshold_type == 'GAP_ONLY']
fig, ax = plt.subplots(figsize=(11, 6))
for (model_name, strategy), values in gap_only.groupby(['model', 'strategy']):
    ax.plot(values.coverage_percentage, values.accepted_accuracy, marker='o', label=f"{model_name.split('/')[-1]} | {strategy}")
ax.axhline(0.90, color='black', linestyle='--', linewidth=1, label='90% precision target')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Accuracy among accepted suggestions')
ax.set_title('Suggestion precision versus coverage')
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Books custom-category learning curve

In [ ]:
display(custom_category_results)
fig, ax = plt.subplots(figsize=(8, 5))
for model_name, values in custom_category_results.groupby('model'):
    ax.plot(values.confirmed_examples, values.books_top1_accuracy, marker='o', label=model_name.split('/')[-1])
ax.set_xticks([0, 1, 3, 5])
ax.set_ylim(0, 1.05)
ax.set_xlabel('Confirmed Books examples')
ax.set_ylabel('Books Top-1 accuracy')
ax.set_title('Custom category improvement without retraining')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## V2 low-confidence and error examples

In [ ]:
low_confidence_v2 = v2_predictions.sort_values('score_gap')[[
    'canonical_id', 'text', 'category', 'predicted_category', 'top1_score',
    'top2_category', 'top2_score', 'score_gap', 'model', 'strategy',
]]
display(low_confidence_v2.head(30))
display(v2_errors.sort_values('score_gap').head(30))